In [1]:
#!pip install torchvision
#!pip install facenet_pytorch

import librosa
import librosa.display
import numpy as np
import pandas as pd
import os
from typing import List, Tuple, Dict
import matplotlib.pyplot as plt
import random
import cv2
# from vgg16 import VGG16_LSTM
import pickle
import torch.nn as nn
import torch
import datetime
import torch.nn.functional as F
import torchvision.models as models
from torchvision import transforms, utils, models
from torch.utils.data import Dataset, DataLoader
# from facenet_pytorch import MTCNN
import logging
import torch
import torch.nn as nn
# from vivit import ViViTBackbone
import gc
from VideoSwinTransformer import SwinTransformer3D
from einops import rearrange
from tqdm import tqdm
# from kw_r2plus1d_video_swin_transformer import SwinTransformer3D
batchsz = 4
frame_nums=15
start_epoch = 0
test_flage=False
model_name ="fullshot"
logger = logging.getLogger(__name__)
logger.setLevel(logging.DEBUG)
formatter = logging.Formatter('[%(asctime)s]::%(module)s::%(levelname)s::%(message)s')
streamHandler = logging.StreamHandler()
streamHandler.setFormatter(formatter)
fileHandler = logging.FileHandler('./LOG/personalityLog.log')
fileHandler.setFormatter(formatter)
logger.addHandler(streamHandler)
logger.addHandler(fileHandler)
USE_CUDA = torch.cuda.is_available()
# device = torch.device('cuda:1' if USE_CUDA else 'cpu')


/home/oem/anaconda3/envs/qx/lib/python3.10/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [2]:

logger.debug('============={}+Video Vision Transformer Strat!============='.format(model_name))
USE_CUDA = torch.cuda.is_available()
print(USE_CUDA)

device = torch.device('cuda:2' if USE_CUDA else 'cpu')
print('학습을 진행하는 기기:', device)

[2025-03-17 17:41:12,229]::2853905153::DEBUG::=============fullshot+Video Vision Transformer Strat!=============


True
학습을 진행하는 기기: cuda:2


In [3]:
def getdata(pathname,datatype):
    if datatype=="train":
        final_data_set=[]
        output_file=open(pathname, "rb")
        for i in range(1):
            # train_data_set.extend(pickle.load(output_file))
            final_data_set.extend(pickle.load(output_file))
            # print(i)
        return final_data_set
    elif datatype=="valid":
        final_data_set=[]
        output_file=open(pathname, "rb")
        for i in range(1):
            valid_data_set=[]
            final_data_set.extend(pickle.load(output_file))
            del valid_data_set
        return final_data_set
print(1)

1


In [4]:
def reshape_to_expected_input(dataset: List[Tuple[ np.ndarray, np.ndarray]]) -> Tuple[
     np.ndarray, np.ndarray]:
    x0_list = []
    x1_list = []
    for i in range(0, len(dataset)):
        x0_list.append(dataset[i][0])
        x1_list.append(dataset[i][1])
    return (np.stack(x0_list), np.stack(x1_list))

In [5]:

save_model_file_path = './save_swintransformer_folder/{}_{}.{}'
print(1)

1


In [6]:
validation_set_data=getdata("/data/qx/datafrom136/data/15Framesfullandfaceandaudio/full/valid_set.dat","valid")
# validation_set_data=getdata("/home/ssrlab/qx/video-swin-transformer-pytorch/data/15Frames/margin011/valid_set.dat","valid")

# validation_set_data=reshape_to_expected_input(validation_set_data)
print(1)

1


In [7]:
train_set_data=getdata("/data/qx/datafrom136/data/15Framesfullandfaceandaudio/full/train_set.dat","train")
# train_set_data=getdata("/home/ssrlab/qx/video-swin-transformer-pytorch/data/15Frames/margin011/train_set.dat","train")

# train_set_data=reshape_to_expected_input(train_set_data)
print(1)

1


In [8]:
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor(),
])
print(1)

1


In [9]:
class ChalearnDataset(Dataset):
    def __init__(self,fullshot,tagdata,transform=None):
        self.fullshot=fullshot
        self.tagdata=tagdata
        self.transform = transform  # 표준화 여부
    def __len__(self):
        return len(self.fullshot)

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()#텐서의 경우 목록으로 돌아가기
        fullshot=self.fullshot[idx]
        fullshot=torch.FloatTensor(fullshot)
        big_five_sorces=self.tagdata[idx]
        big_five_sorces = torch.FloatTensor(big_five_sorces)
        return fullshot,big_five_sorces
print(1)

1


In [10]:
num_workerssz = 20
lr = 3e-05
epochs = 120
print(1)

1


In [11]:
train_set_data = ChalearnDataset(fullshot = train_set_data[1],tagdata=train_set_data[2],transform=transform)
val_set_data = ChalearnDataset(fullshot = validation_set_data[1],tagdata=validation_set_data[2],transform=transform)
train_dataloader = DataLoader(dataset=train_set_data, batch_size=batchsz, shuffle=True, num_workers=num_workerssz)
val_dataloader = DataLoader(dataset=val_set_data, batch_size=batchsz, shuffle=True, num_workers=num_workerssz)
max_value=0
print(1)

1


In [12]:
model=SwinTransformer3D()#num_frames=frame_nums,batchsz = batchsz
model.to(device)
criterion = torch.nn.L1Loss().to(device)  # 손실함수
optimizer = torch.optim.AdamW(model.parameters(), lr=lr) # 옵티마이저

if test_flage==True:
    checkpoint=torch.load(save_model_file_path.format('model',start_epoch,'pth'), map_location=device)
    model.load_state_dict(checkpoint["model"])
    optimizer.load_state_dict(checkpoint["optimizer"])
    # criterion.load_state_dict(checkpoint["loss"])
train_avg_loss0=[]
val_avg_loss0=[]
print(1)

/home/oem/anaconda3/envs/qx/lib/python3.10/site-packages/torch/functional.py:507: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3549.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


1


In [13]:
from torchinfo import summary

summary(model, input_size = ((4, 3,15, 224, 224)), col_names = ['input_size','output_size','num_params'])

Layer (type:depth-idx)                             Input Shape               Output Shape              Param #
SwinTransformer3D                                  [4, 3, 15, 224, 224]      [4, 5]                    2,236,416
├─VideoResNet: 1-1                                 [4, 3, 15, 224, 224]      [4, 512, 2, 14, 14]       205,200
│    └─R2Plus1dStem: 2-1                           [4, 3, 15, 224, 224]      [4, 64, 15, 112, 112]     --
│    │    └─Conv3d: 3-1                            [4, 3, 15, 224, 224]      [4, 45, 15, 112, 112]     6,615
│    │    └─BatchNorm3d: 3-2                       [4, 45, 15, 112, 112]     [4, 45, 15, 112, 112]     90
│    │    └─ReLU: 3-3                              [4, 45, 15, 112, 112]     [4, 45, 15, 112, 112]     --
│    │    └─Conv3d: 3-4                            [4, 45, 15, 112, 112]     [4, 64, 15, 112, 112]     8,640
│    │    └─BatchNorm3d: 3-5                       [4, 64, 15, 112, 112]     [4, 64, 15, 112, 112]     128
│    │    └─ReLU: 3-6 

In [14]:
def extract_ocean(data,batch_size):
    o=[]
    c=[]
    e=[]
    a=[]
    n=[]
    if batch_size >1:
        for i in range(batch_size):
            o.append(data[i][0].item())
            c.append(data[i][1].item())
            e.append(data[i][2].item())
            a.append(data[i][3].item())
            n.append(data[i][4].item())
        o = torch.tensor(o) 
        c = torch.tensor(c) 
        e = torch.tensor(e) 
        a = torch.tensor(a) 
        n = torch.tensor(n) 
    elif batch_size == 1:
        data=data.squeeze()
        for i in range(batch_size):
            o.append(data[0].item())
            c.append(data[1].item())
            e.append(data[2].item())
            a.append(data[3].item())
            n.append(data[4].item())
            
        o = torch.tensor(o) 
        c = torch.tensor(c) 
        e = torch.tensor(e) 
        a = torch.tensor(a) 
        n = torch.tensor(n)
    return o,c,e,a,n

In [ ]:
with torch.cuda.device(2):
    max_value = float('inf')
    for i in range(start_epoch, epochs):
        train_avg_loss = 0
        val_avg_loss = 0
        train_avg_o_loss = 0
        train_avg_c_loss = 0
        train_avg_e_loss = 0
        train_avg_a_loss = 0
        train_avg_n_loss = 0
        val_avg_o_loss = 0
        val_avg_c_loss = 0
        val_avg_e_loss = 0
        val_avg_a_loss = 0
        val_avg_n_loss = 0
        for fullshot,big_five_data in tqdm(train_dataloader):
            fullshot=fullshot.permute(0,4,1,2,3)
            big_five_data=big_five_data.permute(0,2,1)
            big_five_data=big_five_data.squeeze()
            # fullshot=imagecrop(fullshot,batchsz)
            fullshot = fullshot.to(device)
            big_five_data = big_five_data.to(device)
            optimizer.zero_grad()  # 기울기가 0이 됩니다.
            hypothesis = model(fullshot) # 모델의 예측 결과를 저장합니다.
            print(hypothesis)
            print(big_five_data)
            loss = criterion(hypothesis, big_five_data)  # 예측된 결과와 실제 태그 사이의 손실 값을 저장합니다.
            loss.backward()  # 역방향 전파입니다. 
            optimizer.step()  
            train_avg_loss += loss  
        train_avg_loss=train_avg_loss/len(train_dataloader)
        with torch.no_grad():#validate
            for fullshot,big_five_data in tqdm(val_dataloader):
                fullshot=fullshot.permute(0,4,1,2,3)
                # fullshot=imagecrop(fullshot,batchsz)
                big_five_data=big_five_data.permute(0,2,1)
                big_five_data=big_five_data.squeeze()
                fullshot = fullshot.to(device)
                big_five_data = big_five_data.to(device)
                hypothesis = model(fullshot)
                val_loss = criterion(hypothesis, big_five_data)
                hypothesiso,hypothesisc,hypothesise,hypothesisa,hypothesisn=extract_ocean(hypothesis,len(fullshot))
                big_five_datao,big_five_datac,big_five_datae,big_five_dataa,big_five_datan=extract_ocean(big_five_data,len(fullshot))
                val_o_loss = criterion(hypothesiso, big_five_datao) 
                val_c_loss = criterion(hypothesisc, big_five_datac) 
                val_e_loss = criterion(hypothesise, big_five_datae) 
                val_a_loss = criterion(hypothesisa, big_five_dataa) 
                val_n_loss = criterion(hypothesisn, big_five_datan)  
                val_avg_loss += val_loss
                val_avg_o_loss += val_o_loss
                val_avg_c_loss += val_c_loss
                val_avg_e_loss += val_e_loss
                val_avg_a_loss += val_a_loss
                val_avg_n_loss += val_n_loss
            val_avg_loss=val_avg_loss/len(val_dataloader)
            val_avg_o_loss=val_avg_o_loss/len(val_dataloader)
            val_avg_c_loss=val_avg_c_loss/len(val_dataloader)
            val_avg_e_loss=val_avg_e_loss/len(val_dataloader)
            val_avg_a_loss=val_avg_a_loss/len(val_dataloader)
            val_avg_n_loss=val_avg_n_loss/len(val_dataloader)
        torch.cuda.empty_cache()
        # if (1-val_avg_loss)>=(1-max_value):
        start_epoch+=1
        if val_avg_loss < max_value:
            max_value=val_avg_loss
            torch.save({
                'epoch': i+1,
                'model': model.state_dict(),
                'optimizer': optimizer.state_dict(),
                'loss': val_avg_loss,
            }, save_model_file_path.format('model',start_epoch,'pth'))
        logger.debug('Epoch: {} , 1-MAE: {:.4f} , 1-Training Loss: {:.4f} , 1-val_avg_o_loss:{:.4f} , 1-val_avg_c_loss:{:.4f} , 1-val_avg_e_loss:{:.4f} , 1-val_avg_a_loss:{:.4f} , 1-val_avg_n_loss:{:.4f}'.format(i+1, 1-val_avg_loss, 1-train_avg_loss,1-val_avg_o_loss, 1-val_avg_c_loss, 1-val_avg_e_loss, 1-val_avg_a_loss,1-val_avg_n_loss))
    

  0%|          | 0/1498 [00:00<?, ?it/s]

tensor([[ 0.0575, -0.0168,  0.3209,  0.0391, -0.3999],
        [-0.2929, -0.0711, -0.0948,  0.2947,  0.1843],
        [-0.0097,  0.2369,  0.3855,  0.0302, -0.3882],
        [ 0.1114,  0.0882,  0.3166,  0.1967, -0.4422]], device='cuda:2',
       grad_fn=<SqueezeBackward0>)
tensor([[0.4673, 0.5000, 0.6484, 0.6408, 0.5333],
        [0.5327, 0.6979, 0.6044, 0.6505, 0.5222],
        [0.5327, 0.4479, 0.4835, 0.6408, 0.7000],
        [0.2897, 0.3958, 0.3187, 0.4078, 0.4667]], device='cuda:2')


  0%|          | 1/1498 [00:03<1:32:32,  3.71s/it]

tensor([[0.6247, 0.6052, 0.3845, 0.8316, 0.2603],
        [0.7353, 0.5722, 0.7384, 1.0032, 0.3087],
        [0.3606, 0.6332, 0.7315, 1.0413, 0.1507],
        [0.5659, 0.6323, 0.6980, 1.0532, 0.1182]], device='cuda:2',
       grad_fn=<SqueezeBackward0>)
tensor([[0.4206, 0.4792, 0.4615, 0.5631, 0.6778],
        [0.4486, 0.4062, 0.5495, 0.3689, 0.4000],
        [0.5421, 0.5729, 0.6813, 0.6408, 0.6444],
        [0.1589, 0.1458, 0.3297, 0.2330, 0.3333]], device='cuda:2')


  0%|          | 3/1498 [00:04<26:59,  1.08s/it]  

...

  1%|          | 11/1498 [00:05<06:48,  3.64it/s]

tensor([[0.2025, 0.3185, 0.4676, 0.5170, 0.4242],
        [0.1724, 0.3519, 0.5999, 0.5556, 0.2940],
        [0.0883, 0.3940, 0.5238, 0.6794, 0.3532],
        [0.1128, 0.2840, 0.5441, 0.5405, 0.3176]], device='cuda:2',
       grad_fn=<SqueezeBackward0>)
tensor([[0.4393, 0.4792, 0.5275, 0.6019, 0.5444],
        [0.3645, 0.3958, 0.4615, 0.3107, 0.4889],
        [0.7196, 0.6771, 0.7473, 0.6214, 0.7444],
        [0.2617, 0.2604, 0.3516, 0.1553, 0.3333]], device='cuda:2')


  1%|          | 12/1498 [00:06<06:31,  3.79it/s]

tensor([[ 0.2436, -0.1617,  0.9282,  0.5657,  0.4821],
        [ 0.4206,  0.1748,  0.5136,  0.5985,  0.3338],
        [ 0.2387,  0.1334,  0.8576,  0.6188,  0.5940],
        [ 0.5877,  0.2064,  0.6976,  0.3742,  0.4909]], device='cuda:2',
       grad_fn=<SqueezeBackward0>)
tensor([[0.1869, 0.3021, 0.3297, 0.3883, 0.4333],
        [0.0935, 0.1146, 0.1648, 0.3010, 0.1556],
        [0.8411, 0.7917, 0.7912, 0.7184, 0.7333],
        [0.5701, 0.6042, 0.5824, 0.6019, 0.6444]], device='cuda:2')


  1%|          | 13/1498 [00:06<06:16,  3.95it/s]

In [ ]:
# from torchinfo import summary

# summary(model, input_size = (4,15,4,3,224,224))